# Learning Word Embeddings

In [2]:
# ============================================================================
# Imports
# ============================================================================

# lab2: Network construction and visualization
# lab6: Data preparation and embedding training
import sys
import importlib
from pathlib import Path
# add CW2 (parent of embedding_improvements) to path
sys.path.append(str(Path().resolve().parent))

import lab6
import lab2

importlib.reload(lab6)
importlib.reload(lab2)

from lab6 import prepare_visual_genome_text, train_embeddings, ranking_embeddings_signal_to_noise
from lab2 import process_text_network, visualize_network
from lab6 import analyze_embeddings, visualize_embeddings, SkipGramModel
import torch
import os
import numpy as np
from lab6 import train_embeddings
from IPython.display import Image, display

In [2]:
# ============================================================================
# Step 1: Download and Prepare Text Data
# ============================================================================
# This downloads the Visual Genome region descriptions (JSON), extracts it,
# and converts all captions into a single plain text file for processing.
# Output: vg_text.txt (~80MB of text)

# zip_url = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/region_descriptions.json.zip"
# text_file = prepare_visual_genome_text(zip_url)

# print(f"\n✓ Text data ready: {text_file}")

text_file = "../vg_text.txt"  # pre-downloaded for convenience


In [3]:
# ============================================================================
# Step 2: Build Word Co-occurrence Network (will take a few minutes)
# ============================================================================
# Using your process_text_network() function from Lab 2
# This tokenizes the text, counts word co-occurrences, and builds a network
# where nodes are words and edges connect words that appear near each other.
#
# rare_threshold: Filters out uncommon words (keeps top ~0.025% by frequency)
# This gives us ~500 most common words to work with.

print("\n" + "="*80)
print("BUILDING TEXT NETWORK")
print("="*80)

network_data = process_text_network(
    text_file,
    rare_threshold=0.00025,  # Keep only very common tokens
    verbose=False
)

print(f"\n✓ Network built: {network_data['graph'].number_of_nodes():,} nodes, "
      f"{network_data['graph'].number_of_edges():,} edges")


BUILDING TEXT NETWORK

✓ Network built: 458 nodes, 50,128 edges


### Phase 1: Baseline Training

In [27]:


print("\n" + "="*80)
print("🚀 STARTING TRAINING RUN")
print("This may take a few minutes. We are running the full pipeline...")
print("="*80)

phase_dir = "experiments/phase1_baseline"
os.makedirs(phase_dir, exist_ok=True)

baseline_config = {
    'embedding_dim': 64,
    'batch_size': 2048,
    'epochs': 20,
    'learning_rate': 0.001,
    'num_negative': 8,
    'validation_fraction':0.1,
    'context_size': 4,
    'dropout': 0.3,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'exclude_all_contexts': True,
    'patience': 5,
    'device': None
}

results = train_embeddings(
    network_data=network_data,
    **baseline_config,
    save_path=f"{phase_dir}/baseline.pth",
)
# --- Training Summary ---
nodes = results['nodes']
embeddings = results['embeddings']

print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)
print(f"Learned embeddings for {len(nodes):,} words")
print(f"Embedding dimension: {embeddings.shape[1]}")


🚀 STARTING TRAINING RUN
This may take a few minutes. We are running the full pipeline...

🔧 PUNCTUATION FILTER:
  Removed: {'<RARE>', '.', ',', "'"}
  Nodes: 457 → 454
  Edges: 49,823 → 48,463

Train edges: 43,616, Val edges: 4,847


/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 8
  Total samples per epoch: 1,850,958

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 8
  Total samples per epoch: 1,850,958

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

Training on mps
Vocab: 454, Embed dim: 64, Context: 4, Negatives: 8
Regularization: dropout=0.3, weight_decay=0.0001, label_smoothing=0.1


Epoch 01  train=5.6614  val=4.1081  lr=0.001000
  → Best model (val_loss=4.1081), saved to experiments/phase1_baseline/baseline.pth


Epoch 02  train=4.3420  val=3.3620  lr=0.001000
  → Best model (val_loss=3.3620), saved to experiments/phase1_baseline/baseline.pth


...

Epoch 07  train=4.2821  val=3.3582  lr=0.001000


Epoch 08  train=4.2815  val=3.3568  lr=0.001000


Epoch 09  train=4.2814  val=3.3577  lr=0.000500


Epoch 10  train=4.2813  val=3.3578  lr=0.000500


Epoch 11  train=4.2811  val=3.3573  lr=0.000500

Early stopping at epoch 11

Saved loss plot to experiments/phase1_baseline/baseline_training_loss.png

✅ TRAINING COMPLETE
Learned embeddings for 454 words
Embedding dimension: 64


### Phase 1: Baseline Analysis

In [4]:


save_dir = "./experiments/phase1_baseline"
for filename in sorted(os.listdir(save_dir)):
    if not filename.endswith(".pth"):
        continue
    # --- 1. Load Best Model ---
    # We load the model, nodes, and embeddings saved by train_embeddings
    model_path = os.path.join(save_dir, filename)
    prefix = os.path.splitext(filename)[0]  # Remove .pth extension
    output_txt = os.path.join(save_dir, f"{prefix}_results.txt")

    print(f"\nProcessing {filename}")
    print(f"→ Saving output to {output_txt}")

    original_stdout = sys.stdout
    with open(output_txt, "w") as f:
        sys.stdout = f  # Change the standard output to the file we created.
        if os.path.exists(model_path):
            print(f"{save_dir}: Loading best model from {model_path}...")
            checkpoint = torch.load(model_path)
            nodes = checkpoint['nodes']

            # Reconstruct model to get final (non-detached) embeddings
            # Note: We use .get_embeddings() which returns the center_embeddings
            model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
            model.load_state_dict(checkpoint['model_state_dict'])
            embeddings = model.get_embeddings()

            print(f"✅ Loaded embeddings for {len(nodes):,} words")
        else:
            print("❌ Error: No saved model found!")
            print("Please run the training cell above before proceeding.")
            # Raise an error to stop execution if the model is missing
            raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

        # --- 2. Qualitative Analysis ---
        # Now we use the helper functions to probe what the model learned.
        # We'll check for nearest neighbors (semantics) and analogies (linear structure).
        analyze_embeddings(
            nodes=nodes,
            embeddings=embeddings,

            # --- Nearest neighbors ---
            # Probe visual semantics, attributes, and compositional structure
            similarity_examples=[
                # People and roles
                "man", "woman", "child", "boy", "girl", "person",
                # Scene elements
                "tree", "building", "sky", "street", "car", "table",
                # Colors and materials
                "red", "blue", "green", "black", "white", "wooden", "metal",
                # Actions
                "walking", "sitting", "holding", "riding", "standing",
                # Textures / objects
                "water", "grass", "sand", "snow", "wall"
            ],

            # --- Analogies ---
            # a : b :: c : ?
            #  → test if embedding geometry captures color, role, size, material, and action relations
            analogy_examples=[
                # Color analogies
                ("red", "apple", "yellow"),        # → banana?
                ("blue", "sky", "green"),          # → grass?
                ("white", "snow", "brown"),        # → dirt or wood?

                # Role analogies
                ("man", "woman", "boy"),           # → girl?
                ("boy", "girl", "man"),            # → woman?
                ("person", "hat", "hand"),         # → glove?

                # Size / quantity / attribute analogies
                ("long", "short", "tall"),         # → short?
                ("one", "two", "three"),           # → four?

                # Action and object use
                ("dog", "walking", "cat"),         # → sitting?
                ("person", "riding", "boat"),      # → sitting or water?
                ("pizza", "eating", "umbrella"),   # → holding?

                # Material / context
                ("metal", "car", "wooden"),        # → table?
                ("water", "boat", "road"),         # → car?
            ],

            # --- Semantic clusters ---
            # seeds chosen to reveal structure in scene-object relationships
            cluster_seeds=[
                "red", "blue", "green",          # color cluster
                "man", "woman", "child",         # human cluster
                "dog", "cat", "horse", "bird",   # animal cluster
                "building", "house", "street",   # architecture/scene
                "sky", "clouds", "water", "grass", # natural elements
                "car", "bus", "bicycle", "train" # vehicles
            ]
        )

        # --- 3. t-SNE Visualization ---
        # We'll create a 2D plot of the *most frequent* words
        # NOTE: Visualizing all 10k+ words is too slow and unreadable.
        # The function defaults to the top 200, we'll use 300.
        print("\n" + "="*80)
        print("VISUALIZING EMBEDDINGS")
        print("="*80)
        print("Generating t-SNE plot for the top 300 words...")
        print("(This may take a moment...)")

        tsne_file = os.path.join(save_dir, f"{prefix}_tsne.png")
        visualize_embeddings(
            nodes,
            embeddings,
            output_file=tsne_file,
            sample_size=len(nodes),
            annotate=True
        )
    sys.stdout = original_stdout  # Reset standard output to original value


Processing baseline.pth
→ Saving output to ./experiments/phase1_baseline/baseline_results.txt


In [29]:
from lab6 import run_tests

print("="*80)
print("🧪 RUNNING UNIT TESTS")
print("="*80)
print("This will test your implementations of SkipGramDataset and SkipGramModel.")
print("It's crucial that you test your implementations in this lab.")

# This function will run all tests defined in the lab6.py file
# Look for an "OK" or "PASSED" summary at the end.
success = run_tests()

print("\n" + "="*80)
if success:
    print("✅ All tests passed! You are ready to move on to the next step.")
else:
    print("❌ Some tests failed. Please review the errors above and fix your code.")
print("="*80)

test_build_contexts_basic (lab6.TestSkipGramDataset.test_build_contexts_basic)
Test that contexts are built correctly for connected nodes. ... ok
test_build_contexts_isolated_node (lab6.TestSkipGramDataset.test_build_contexts_isolated_node)
Test that isolated nodes have empty context. ... ok
test_build_contexts_multi_hop (lab6.TestSkipGramDataset.test_build_contexts_multi_hop)
Test context building with context_size > 1. ... ok
test_empty_graph (lab6.TestSkipGramDataset.test_empty_graph)
Test handling of graph with no edges. ... ok
test_generate_weighted_pairs_count (lab6.TestSkipGramDataset.test_generate_weighted_pairs_count)
Test that correct number of pairs are generated. ... ok
test_generate_weighted_pairs_symmetry (lab6.TestSkipGramDataset.test_generate_weighted_pairs_symmetry)
Test that pairs are bidirectional for undirected graph. ... ok
test_getitem_exclusions (lab6.TestSkipGramDataset.test_getitem_exclusions)
Test that negatives don't include center or its contexts. ... ok
tes

🧪 RUNNING UNIT TESTS
This will test your implementations of SkipGramDataset and SkipGramModel.
It's crucial that you test your implementations in this lab.
RUNNING SKIP-GRAM UNIT TESTS

✅ ALL TESTS PASSED!
Total tests run: 24

✅ All tests passed! You are ready to move on to the next step.


### Phase 2: Embedding Size Training

In [30]:
phase_dir = "experiments/phase2_dimension"
os.makedirs(phase_dir, exist_ok=True)

dims = [128, 256, 384, 512]

for dim in dims:
    baseline_config = {
        'embedding_dim': dim,
        'batch_size': 2048,
        'epochs': 20,
        'learning_rate': 0.001,
        'num_negative': 8,
        'validation_fraction':0.1,
        'context_size': 4,
        'dropout': 0.3,
        'weight_decay': 1e-4,
        'label_smoothing': 0.1,
        'exclude_all_contexts': True,
        'patience': 5,
        'device': None
    }
    results = train_embeddings(
        network_data=network_data,
        **baseline_config,
        save_path=f"{phase_dir}/dim_{dim}.pth",
    )


🔧 PUNCTUATION FILTER:
  Removed: {'<RARE>', '.', ',', "'"}
  Nodes: 457 → 454
  Edges: 49,823 → 48,463

Train edges: 43,616, Val edges: 4,847

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 8
  Total samples per epoch: 1,850,958

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000


/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 8
  Total samples per epoch: 1,850,958

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

Training on mps
Vocab: 454, Embed dim: 128, Context: 4, Negatives: 8
Regularization: dropout=0.3, weight_decay=0.0001, label_smoothing=0.1


Epoch 01  train=5.3514  val=3.4792  lr=0.001000
  → Best model (val_loss=3.4792), saved to experiments/phase2_dimension/dim_128.pth


Epoch 02  train=4.2812  val=3.3553  lr=0.001000
  → Best model (val_loss=3.3553), saved to experiments/phase2_dimension/dim_128.pth


...

Epoch 04  train=4.2710  val=3.3577  lr=0.001000


Epoch 05  train=4.2619  val=3.3623  lr=0.001000


Epoch 06  train=4.2546  val=3.3641  lr=0.001000


Epoch 07  train=4.2482  val=3.3632  lr=0.000500


Epoch 08  train=4.2422  val=3.3640  lr=0.000500

Early stopping at epoch 8

Saved loss plot to experiments/phase2_dimension/dim_512_training_loss.png


### Phase 2: Embedding Size Analysis

In [31]:
save_dir = "./experiments/phase2_dimension"
for filename in sorted(os.listdir(save_dir)):
    if not filename.endswith(".pth"):
        continue
    # --- 1. Load Best Model ---
    # We load the model, nodes, and embeddings saved by train_embeddings
    model_path = os.path.join(save_dir, filename)
    prefix = os.path.splitext(filename)[0]  # Remove .pth extension
    output_txt = os.path.join(save_dir, f"{prefix}_results.txt")

    print(f"\nProcessing {filename}")
    print(f"→ Saving output to {output_txt}")

    original_stdout = sys.stdout
    with open(output_txt, "w") as f:
        sys.stdout = f  # Change the standard output to the file we created.
        if os.path.exists(model_path):
            print(f"{save_dir}: Loading best model from {model_path}...")
            checkpoint = torch.load(model_path)
            nodes = checkpoint['nodes']

            # Reconstruct model to get final (non-detached) embeddings
            # Note: We use .get_embeddings() which returns the center_embeddings
            model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
            model.load_state_dict(checkpoint['model_state_dict'])
            embeddings = model.get_embeddings()

            print(f"✅ Loaded embeddings for {len(nodes):,} words")
        else:
            print("❌ Error: No saved model found!")
            print("Please run the training cell above before proceeding.")
            # Raise an error to stop execution if the model is missing
            raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

        # --- 2. Qualitative Analysis ---
        # Now we use the helper functions to probe what the model learned.
        # We'll check for nearest neighbors (semantics) and analogies (linear structure).
        analyze_embeddings(
            nodes=nodes,
            embeddings=embeddings,

            # --- Nearest neighbors ---
            # Probe visual semantics, attributes, and compositional structure
            similarity_examples=[
                # People and roles
                "man", "woman", "child", "boy", "girl", "person",
                # Scene elements
                "tree", "building", "sky", "street", "car", "table",
                # Colors and materials
                "red", "blue", "green", "black", "white", "wooden", "metal",
                # Actions
                "walking", "sitting", "holding", "riding", "standing",
                # Textures / objects
                "water", "grass", "sand", "snow", "wall"
            ],

            # --- Analogies ---
            # a : b :: c : ?
            #  → test if embedding geometry captures color, role, size, material, and action relations
            analogy_examples=[
                # Color analogies
                ("red", "apple", "yellow"),        # → banana?
                ("blue", "sky", "green"),          # → grass?
                ("white", "snow", "brown"),        # → dirt or wood?

                # Role analogies
                ("man", "woman", "boy"),           # → girl?
                ("boy", "girl", "man"),            # → woman?
                ("person", "hat", "hand"),         # → glove?

                # Size / quantity / attribute analogies
                ("long", "short", "tall"),         # → short?
                ("one", "two", "three"),           # → four?

                # Action and object use
                ("dog", "walking", "cat"),         # → sitting?
                ("person", "riding", "boat"),      # → sitting or water?
                ("pizza", "eating", "umbrella"),   # → holding?

                # Material / context
                ("metal", "car", "wooden"),        # → table?
                ("water", "boat", "road"),         # → car?
            ],

            # --- Semantic clusters ---
            # seeds chosen to reveal structure in scene-object relationships
            cluster_seeds=[
                "red", "blue", "green",          # color cluster
                "man", "woman", "child",         # human cluster
                "dog", "cat", "horse", "bird",   # animal cluster
                "building", "house", "street",   # architecture/scene
                "sky", "clouds", "water", "grass", # natural elements
                "car", "bus", "bicycle", "train" # vehicles
            ]
        )

        # --- 3. t-SNE Visualization ---
        # We'll create a 2D plot of the *most frequent* words
        # NOTE: Visualizing all 10k+ words is too slow and unreadable.
        # The function defaults to the top 200, we'll use 300.
        print("\n" + "="*80)
        print("VISUALIZING EMBEDDINGS")
        print("="*80)
        print("Generating t-SNE plot for the top 300 words...")
        print("(This may take a moment...)")

        tsne_file = os.path.join(save_dir, f"{prefix}_tsne.png")
        visualize_embeddings(
            nodes,
            embeddings,
            output_file=tsne_file,
            sample_size=len(nodes),
            annotate=True
        )
    sys.stdout = original_stdout  # Reset standard output to original value


Processing dim_128.pth
→ Saving output to ./experiments/phase2_dimension/dim_128_results.txt

Processing dim_256.pth
→ Saving output to ./experiments/phase2_dimension/dim_256_results.txt

Processing dim_384.pth
→ Saving output to ./experiments/phase2_dimension/dim_384_results.txt

Processing dim_512.pth
→ Saving output to ./experiments/phase2_dimension/dim_512_results.txt


## Phase 3: Context Size

### 3A: Training

In [39]:
phase_dir = "experiments/phase3_contextSize"
os.makedirs(phase_dir, exist_ok=True)

context_sizes = [1, 2, 4, 6]

for c in context_sizes:
    baseline_config = {
        'embedding_dim': 512,
        'batch_size': 2048,
        'epochs': 20,
        'learning_rate': 0.001,
        'num_negative': 8,
        'validation_fraction':0.1,
        'context_size': c,
        'dropout': 0.3,
        'weight_decay': 1e-4,
        'label_smoothing': 0.1,
        'exclude_all_contexts': True,
        'patience': 5,
        'device': None
    }
    results = train_embeddings(
        network_data=network_data,
        **baseline_config,
        save_path=f"{phase_dir}/context_{c}.pth",
    )

/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



🔧 PUNCTUATION FILTER:
  Removed: {'<RARE>', '.', ',', "'"}
  Nodes: 457 → 454
  Edges: 49,823 → 48,463

Train edges: 43,616, Val edges: 4,847

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 87,232
  Negatives per positive: 8
  Total samples per epoch: 785,088

  Weight distribution:
    Min: 0.015002
    Mean: 0.080676
    Median: 0.031823
    Max: 1.000000

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 9,694
  Negatives per positive: 8
  Total samples per epoch: 87,246

  Weight distribution:
    Min: 0.015458
    Mean: 0.083116
    Median: 0.034565
    Max: 1.000000

Training on mps
Vocab: 454, Embed dim: 512, Context: 1, Negatives: 8
Regularization: dropout=0.3, weight_decay=0.0001, label_smoothing=0.1


Epoch 01  train=5.6919  val=4.3140  lr=0.001000
  → Best model (val_loss=4.3140), saved to experiments/phase3_contextSize/context_1.pth


Epoch 02  train=4.0951  val=3.4611  lr=0.001000
  → Best model (val_loss=3.4611), saved to experiments/phase3_contextSize/context_1.pth


Epoch 03  train=3.9581  val=3.5306  lr=0.001000


...

Epoch 05  train=4.2631  val=3.3602  lr=0.001000


Epoch 06  train=4.2532  val=3.3649  lr=0.001000


Epoch 07  train=4.2480  val=3.3651  lr=0.001000


Epoch 08  train=4.2378  val=3.3633  lr=0.000500


KeyboardInterrupt: 

### 3B: Analysis

In [40]:
save_dir = "./experiments/phase3_contextSize"
for filename in sorted(os.listdir(save_dir)):
    if not filename.endswith(".pth"):
        continue
    # --- 1. Load Best Model ---
    # We load the model, nodes, and embeddings saved by train_embeddings
    model_path = os.path.join(save_dir, filename)
    prefix = os.path.splitext(filename)[0]  # Remove .pth extension
    output_txt = os.path.join(save_dir, f"{prefix}_results.txt")

    print(f"\nProcessing {filename}")
    print(f"→ Saving output to {output_txt}")

    original_stdout = sys.stdout
    with open(output_txt, "w") as f:
        sys.stdout = f  # Change the standard output to the file we created.
        if os.path.exists(model_path):
            print(f"{save_dir}: Loading best model from {model_path}...")
            checkpoint = torch.load(model_path)
            nodes = checkpoint['nodes']

            # Reconstruct model to get final (non-detached) embeddings
            # Note: We use .get_embeddings() which returns the center_embeddings
            model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
            model.load_state_dict(checkpoint['model_state_dict'])
            embeddings = model.get_embeddings()

            print(f"✅ Loaded embeddings for {len(nodes):,} words")
        else:
            print("❌ Error: No saved model found!")
            print("Please run the training cell above before proceeding.")
            # Raise an error to stop execution if the model is missing
            raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

        # --- 2. Qualitative Analysis ---
        # Now we use the helper functions to probe what the model learned.
        # We'll check for nearest neighbors (semantics) and analogies (linear structure).
        analyze_embeddings(
            nodes=nodes,
            embeddings=embeddings,

            # --- Nearest neighbors ---
            # Probe visual semantics, attributes, and compositional structure
            similarity_examples=[
                # People and roles
                "man", "woman", "child", "boy", "girl", "person",
                # Scene elements
                "tree", "building", "sky", "street", "car", "table",
                # Colors and materials
                "red", "blue", "green", "black", "white", "wooden", "metal",
                # Actions
                "walking", "sitting", "holding", "riding", "standing",
                # Textures / objects
                "water", "grass", "sand", "snow", "wall"
            ],

            # --- Analogies ---
            # a : b :: c : ?
            #  → test if embedding geometry captures color, role, size, material, and action relations
            analogy_examples=[
                # Color analogies
                ("red", "apple", "yellow"),        # → banana?
                ("blue", "sky", "green"),          # → grass?
                ("white", "snow", "brown"),        # → dirt or wood?

                # Role analogies
                ("man", "woman", "boy"),           # → girl?
                ("boy", "girl", "man"),            # → woman?
                ("person", "hat", "hand"),         # → glove?

                # Size / quantity / attribute analogies
                ("long", "short", "tall"),         # → short?
                ("one", "two", "three"),           # → four?

                # Action and object use
                ("dog", "walking", "cat"),         # → sitting?
                ("person", "riding", "boat"),      # → sitting or water?
                ("pizza", "eating", "umbrella"),   # → holding?

                # Material / context
                ("metal", "car", "wooden"),        # → table?
                ("water", "boat", "road"),         # → car?
            ],

            # --- Semantic clusters ---
            # seeds chosen to reveal structure in scene-object relationships
            cluster_seeds=[
                "red", "blue", "green",          # color cluster
                "man", "woman", "child",         # human cluster
                "dog", "cat", "horse", "bird",   # animal cluster
                "building", "house", "street",   # architecture/scene
                "sky", "clouds", "water", "grass", # natural elements
                "car", "bus", "bicycle", "train" # vehicles
            ]
        )

        # --- 3. t-SNE Visualization ---
        # We'll create a 2D plot of the *most frequent* words
        # NOTE: Visualizing all 10k+ words is too slow and unreadable.
        # The function defaults to the top 200, we'll use 300.
        print("\n" + "="*80)
        print("VISUALIZING EMBEDDINGS")
        print("="*80)
        print("Generating t-SNE plot for the top 300 words...")
        print("(This may take a moment...)")

        tsne_file = os.path.join(save_dir, f"{prefix}_tsne.png")
        visualize_embeddings(
            nodes,
            embeddings,
            output_file=tsne_file,
            sample_size=len(nodes),
            annotate=True
        )
    sys.stdout = original_stdout  # Reset standard output to original value


Processing context_1.pth
→ Saving output to ./experiments/phase3_contextSize/context_1_results.txt

Processing context_2.pth
→ Saving output to ./experiments/phase3_contextSize/context_2_results.txt

Processing context_4.pth
→ Saving output to ./experiments/phase3_contextSize/context_4_results.txt

Processing context_6.pth
→ Saving output to ./experiments/phase3_contextSize/context_6_results.txt


## Phase 3: Negative Sampling Strategy

### 3A. Training

In [35]:
phase_dir = "experiments/phase3_num_negatives"
os.makedirs(phase_dir, exist_ok=True)
os.makedirs("experiments/phase3_num_negatives/exclude_True", exist_ok=True)
os.makedirs("experiments/phase3_num_negatives/exclude_False", exist_ok=True)

negatives = [10, 15, 20]
excluding_contexts = [True, False]

for num_negatives in negatives:
    for exclude_context in excluding_contexts:
        baseline_config = {
            'embedding_dim': 512,
            'batch_size': 2048,
            'epochs': 20,
            'learning_rate': 0.001,
            'num_negative': num_negatives,
            'validation_fraction':0.1,
            'context_size': 4,
            'dropout': 0.3,
            'weight_decay': 1e-4,
            'label_smoothing': 0.1,
            'exclude_all_contexts': exclude_context,
            'patience': 5,
            'device': None
        }
        results = train_embeddings(
            network_data=network_data,
            **baseline_config,
            save_path=f"{phase_dir}/exclude_{exclude_context}/neg_{num_negatives}.pth",
        )


🔧 PUNCTUATION FILTER:
  Removed: {'<RARE>', '.', ',', "'"}
  Nodes: 457 → 454
  Edges: 49,823 → 48,463

Train edges: 43,616, Val edges: 4,847

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 10
  Total samples per epoch: 2,262,282

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 10
  Total samples per epoch: 2,262,282

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

Training on mps
Vocab: 454, Embed dim: 512, Context: 4, Negatives: 10
Regularization: dropout=0.3, weight_decay=0.0001, label_smoothing=0.1


Epoch 01  train=5.8159  val=3.6779  lr=0.001000
  → Best model (val_loss=3.6779), saved to experiments/phase3_num_negatives/exclude_True/neg_10.pth


Epoch 02  train=4.9794  val=3.6606  lr=0.001000
  → Best model (val_loss=3.6606), saved to experiments/phase3_num_negatives/exclude_True/neg_10.pth


Epoch 03  train=4.9787  val=3.6627  lr=0.001000


Epoch 04  train=4.9774  val=3.6634  lr=0.001000


...

Epoch 04  train=8.3679  val=4.9496  lr=0.001000


Epoch 05  train=8.3682  val=4.9520  lr=0.001000


Epoch 06  train=8.3636  val=4.9509  lr=0.001000


Epoch 07  train=8.3592  val=4.9483  lr=0.000500


Epoch 08  train=8.3551  val=4.9550  lr=0.000500

Early stopping at epoch 8

Saved loss plot to experiments/phase3_num_negatives/exclude_False/neg_20_training_loss.png


### 3B. Analysis

In [36]:
phase_dir = "./experiments/phase3_num_negatives"
for save_dir in [f"{phase_dir}/exclude_{ec}" for ec in [True, False]]:
    for filename in sorted(os.listdir(save_dir)):
        if not filename.endswith(".pth"):
            continue
        # --- 1. Load Best Model ---
        # We load the model, nodes, and embeddings saved by train_embeddings
        model_path = os.path.join(save_dir, filename)
        prefix = os.path.splitext(filename)[0]  # Remove .pth extension
        output_txt = os.path.join(save_dir, f"{prefix}_results.txt")

        print(f"\nProcessing {filename}")
        print(f"→ Saving output to {output_txt}")

        original_stdout = sys.stdout
        with open(output_txt, "w") as f:
            sys.stdout = f  # Change the standard output to the file we created.
            if os.path.exists(model_path):
                print(f"{save_dir}: Loading best model from {model_path}...")
                checkpoint = torch.load(model_path)
                nodes = checkpoint['nodes']

                # Reconstruct model to get final (non-detached) embeddings
                # Note: We use .get_embeddings() which returns the center_embeddings
                model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
                model.load_state_dict(checkpoint['model_state_dict'])
                embeddings = model.get_embeddings()

                print(f"✅ Loaded embeddings for {len(nodes):,} words")
            else:
                print("❌ Error: No saved model found!")
                print("Please run the training cell above before proceeding.")
                # Raise an error to stop execution if the model is missing
                raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

            # --- 2. Qualitative Analysis ---
            # Now we use the helper functions to probe what the model learned.
            # We'll check for nearest neighbors (semantics) and analogies (linear structure).
            analyze_embeddings(
                nodes=nodes,
                embeddings=embeddings,

                # --- Nearest neighbors ---
                # Probe visual semantics, attributes, and compositional structure
                similarity_examples=[
                    # People and roles
                    "man", "woman", "child", "boy", "girl", "person",
                    # Scene elements
                    "tree", "building", "sky", "street", "car", "table",
                    # Colors and materials
                    "red", "blue", "green", "black", "white", "wooden", "metal",
                    # Actions
                    "walking", "sitting", "holding", "riding", "standing",
                    # Textures / objects
                    "water", "grass", "sand", "snow", "wall"
                ],

                # --- Analogies ---
                # a : b :: c : ?
                #  → test if embedding geometry captures color, role, size, material, and action relations
                analogy_examples=[
                    # Color analogies
                    ("red", "apple", "yellow"),        # → banana?
                    ("blue", "sky", "green"),          # → grass?
                    ("white", "snow", "brown"),        # → dirt or wood?

                    # Role analogies
                    ("man", "woman", "boy"),           # → girl?
                    ("boy", "girl", "man"),            # → woman?
                    ("person", "hat", "hand"),         # → glove?

                    # Size / quantity / attribute analogies
                    ("long", "short", "tall"),         # → short?
                    ("one", "two", "three"),           # → four?

                    # Action and object use
                    ("dog", "walking", "cat"),         # → sitting?
                    ("person", "riding", "boat"),      # → sitting or water?
                    ("pizza", "eating", "umbrella"),   # → holding?

                    # Material / context
                    ("metal", "car", "wooden"),        # → table?
                    ("water", "boat", "road"),         # → car?
                ],

                # --- Semantic clusters ---
                # seeds chosen to reveal structure in scene-object relationships
                cluster_seeds=[
                    "red", "blue", "green",          # color cluster
                    "man", "woman", "child",         # human cluster
                    "dog", "cat", "horse", "bird",   # animal cluster
                    "building", "house", "street",   # architecture/scene
                    "sky", "clouds", "water", "grass", # natural elements
                    "car", "bus", "bicycle", "train" # vehicles
                ]
            )

            # --- 3. t-SNE Visualization ---
            # We'll create a 2D plot of the *most frequent* words
            # NOTE: Visualizing all 10k+ words is too slow and unreadable.
            # The function defaults to the top 200, we'll use 300.
            print("\n" + "="*80)
            print("VISUALIZING EMBEDDINGS")
            print("="*80)
            print("Generating t-SNE plot for the top 300 words...")
            print("(This may take a moment...)")

            tsne_file = os.path.join(save_dir, f"{prefix}_tsne.png")
            visualize_embeddings(
                nodes,
                embeddings,
                output_file=tsne_file,
                sample_size=len(nodes),
                annotate=True
            )
        sys.stdout = original_stdout  # Reset standard output to original value


Processing neg_10.pth
→ Saving output to ./experiments/phase3_num_negatives/exclude_True/neg_10_results.txt

Processing neg_15.pth
→ Saving output to ./experiments/phase3_num_negatives/exclude_True/neg_15_results.txt

Processing neg_20.pth
→ Saving output to ./experiments/phase3_num_negatives/exclude_True/neg_20_results.txt

Processing neg_10.pth
→ Saving output to ./experiments/phase3_num_negatives/exclude_False/neg_10_results.txt

Processing neg_15.pth
→ Saving output to ./experiments/phase3_num_negatives/exclude_False/neg_15_results.txt

Processing neg_20.pth
→ Saving output to ./experiments/phase3_num_negatives/exclude_False/neg_20_results.txt


# Grid Search

## Architecture
Changing embedding_dim, context_size, and num_negative

In [42]:
from itertools import product

phase_dir = "gridSearch/architecture"
os.makedirs(phase_dir, exist_ok=True)

sweep_params = {
    'embedding_dim': [128, 256, 384, 512],
    'num_negative': [8, 13, 18, 23],
    'context_size': [2, 3, 4],
}

fixed_params = {
    'batch_size': 1024,
    'epochs': 20,
    'learning_rate': 0.001,
    'validation_fraction': 0.1,
    'dropout': 0.2,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'exclude_all_contexts': True,
    'patience': 5,
    'device': None
}

for dim, n_neg, ctx in product(
    sweep_params['embedding_dim'],
    sweep_params['num_negative'],
    sweep_params['context_size'],
):
    config = {
        **fixed_params,
        'embedding_dim': dim,
        'num_negative': n_neg,
        'context_size': ctx,
    }
    print(f"Running: dim={dim}, neg={n_neg}, ctx={ctx}")
    results = train_embeddings(
        network_data=network_data,
        **config,
        save_path=f"{phase_dir}/dim_{dim}_neg_{n_neg}_context_{ctx}.pth",
    )



Running: dim=128, neg=8, ctx=2

🔧 PUNCTUATION FILTER:
  Removed: {'<RARE>', '.', ',', "'"}
  Nodes: 457 → 454
  Edges: 49,823 → 48,463

Train edges: 43,616, Val edges: 4,847


/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 8
  Total samples per epoch: 1,850,958

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 139,382
  Negatives per positive: 8
  Total samples per epoch: 1,254,438

  Weight distribution:
    Min: 0.023168
    Mean: 0.080509
    Median: 0.028375
    Max: 1.000000

Training on mps
Vocab: 454, Embed dim: 128, Context: 2, Negatives: 8
Regularization: dropout=0.2, weight_decay=0.0001, label_smoothing=0.1


Epoch 01  train=4.8491  val=3.2118  lr=0.001000
  → Best model (val_loss=3.2118), saved to gridSearch/architecture/dim_128_neg_8_context_2.pth


Epoch 02  train=4.2769  val=3.2110  lr=0.001000
  → Best model (val_loss=3.2110), saved to gridSearch/architecture/dim_128_neg_8_context_2.pth


...

Epoch 02  train=9.3647  val=5.3002  lr=0.001000


Epoch 03  train=9.3633  val=5.3026  lr=0.001000


Epoch 04  train=9.3559  val=5.3100  lr=0.001000


Epoch 05  train=9.3496  val=5.3071  lr=0.000500


Epoch 06  train=9.3473  val=5.3131  lr=0.000500

Early stopping at epoch 6

Saved loss plot to gridSearch/architecture/dim_512_neg_23_context_4_training_loss.png


### Analysis
On two metrics:
- max(max cosine sim - avg cosing sim)
- cifar superclass similarity

In [2]:
save_dir = "./gridSearch/architecture"
all_signal_to_noise = {}

for filename in sorted(os.listdir(save_dir)):
    if not filename.endswith(".pth"):
        continue
    # --- 1. Load Best Model ---
    # We load the model, nodes, and embeddings saved by train_embeddings
    model_path = os.path.join(save_dir, filename)
    prefix = os.path.splitext(filename)[0]  # Remove .pth extension
    output_txt = os.path.join(save_dir + "/results/", f"{prefix}.txt")

    print(f"\nProcessing {filename}")
    print(f"→ Saving output to {output_txt}")

    original_stdout = sys.stdout
    with open(output_txt, "w") as f:
        sys.stdout = f  # Change the standard output to the file we created.
        if os.path.exists(model_path):
            print(f"{save_dir}: Loading best model from {model_path}...")
            checkpoint = torch.load(model_path)
            nodes = checkpoint['nodes']

            # Reconstruct model to get final (non-detached) embeddings
            # Note: We use .get_embeddings() which returns the center_embeddings
            model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
            model.load_state_dict(checkpoint['model_state_dict'])
            embeddings = model.get_embeddings()

            print(f"✅ Loaded embeddings for {len(nodes):,} words")
        else:
            print("❌ Error: No saved model found!")
            print("Please run the training cell above before proceeding.")
            # Raise an error to stop execution if the model is missing
            raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

        # --- 2. Qualitative Analysis ---
        signal_to_noise_metric =  ranking_embeddings_signal_to_noise(nodes, embeddings)
        print(f"Signal to Noise Metric: {signal_to_noise_metric:.4f}")
        all_signal_to_noise[prefix] = signal_to_noise_metric
        # Now we use the helper functions to probe what the model learned.
        # We'll check for nearest neighbors (semantics) and analogies (linear structure).
        analyze_embeddings(
            nodes=nodes,
            embeddings=embeddings,

            # --- Nearest neighbors ---
            # Probe visual semantics, attributes, and compositional structure
            similarity_examples=[
                # People and roles
                "man", "woman", "child", "boy", "girl", "person",
                # Scene elements
                "tree", "building", "sky", "street", "car", "table",
                # Colors and materials
                "red", "blue", "green", "black", "white", "wooden", "metal",
                # Actions
                "walking", "sitting", "holding", "riding", "standing",
                # Textures / objects
                "water", "grass", "sand", "snow", "wall"
            ],

            # --- Analogies ---
            # a : b :: c : ?
            #  → test if embedding geometry captures color, role, size, material, and action relations
            analogy_examples=[
                # Color analogies
                ("red", "apple", "yellow"),        # → banana?
                ("blue", "sky", "green"),          # → grass?
                ("white", "snow", "brown"),        # → dirt or wood?

                # Role analogies
                ("man", "woman", "boy"),           # → girl?
                ("boy", "girl", "man"),            # → woman?
                ("person", "hat", "hand"),         # → glove?

                # Size / quantity / attribute analogies
                ("long", "short", "tall"),         # → short?
                ("one", "two", "three"),           # → four?

                # Action and object use
                ("dog", "walking", "cat"),         # → sitting?
                ("person", "riding", "boat"),      # → sitting or water?
                ("pizza", "eating", "umbrella"),   # → holding?

                # Material / context
                ("metal", "car", "wooden"),        # → table?
                ("water", "boat", "road"),         # → car?
            ],

            # --- Semantic clusters ---
            # seeds chosen to reveal structure in scene-object relationships
            cluster_seeds=[
                "red", "blue", "green",          # color cluster
                "man", "woman", "child",         # human cluster
                "dog", "cat", "horse", "bird",   # animal cluster
                "building", "house", "street",   # architecture/scene
                "sky", "clouds", "water", "grass", # natural elements
                "car", "bus", "bicycle", "train" # vehicles
            ]
        )

        # --- 3. t-SNE Visualization ---
        # We'll create a 2D plot of the *most frequent* words
        # NOTE: Visualizing all 10k+ words is too slow and unreadable.
        # The function defaults to the top 200, we'll use 300.
        print("\n" + "="*80)
        print("VISUALIZING EMBEDDINGS")
        print("="*80)
        print("Generating t-SNE plot for the top 300 words...")
        print("(This may take a moment...)")

        tsne_file = os.path.join(save_dir + "/results", f"{prefix}_tsne.png")
        visualize_embeddings(
            nodes,
            embeddings,
            output_file=tsne_file,
            sample_size=len(nodes),
            annotate=True
        )
    sys.stdout = original_stdout  # Reset standard output to original value


Processing dim_128_neg_13_context_2.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_13_context_2.txt

Processing dim_128_neg_13_context_3.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_13_context_3.txt

Processing dim_128_neg_13_context_4.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_13_context_4.txt

Processing dim_128_neg_18_context_2.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_18_context_2.txt

Processing dim_128_neg_18_context_3.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_18_context_3.txt

Processing dim_128_neg_18_context_4.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_18_context_4.txt

Processing dim_128_neg_23_context_2.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_23_context_2.txt

Processing dim_128_neg_23_context_3.pth
→ Saving output to ./gridSearch/architecture/results/dim_128_neg_23_context_3.txt

Processing dim_

In [3]:
# Top 10 in all_mean_minus_p5, all_cifar_cls_sim -> see if there's anything in common
def top_k(d, k=10):
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:k]

top_signal_to_noise = top_k(all_signal_to_noise, k=20)

print("Top 10 Signal to Noise Metrics:")
for name, score in top_signal_to_noise:
    print(f"{name}: {score:.4f}")       

Top 10 Signal to Noise Metrics:
dim_512_neg_18_context_2: 0.1121
dim_384_neg_13_context_2: 0.0285
dim_256_neg_18_context_2: 0.0247
dim_128_neg_8_context_3: 0.0247
dim_256_neg_8_context_2: 0.0189
dim_384_neg_23_context_4: 0.0164
dim_384_neg_8_context_4: 0.0106
dim_384_neg_8_context_2: 0.0076
dim_128_neg_13_context_2: 0.0045
dim_512_neg_13_context_3: 0.0029
dim_128_neg_18_context_2: 0.0023
dim_128_neg_8_context_2: 0.0023
dim_512_neg_18_context_3: 0.0022
dim_256_neg_8_context_4: 0.0015
dim_128_neg_8_context_4: 0.0014
dim_512_neg_18_context_4: 0.0010
dim_512_neg_8_context_3: 0.0008
dim_128_neg_23_context_2: 0.0007
dim_384_neg_13_context_4: 0.0006
dim_256_neg_18_context_3: 0.0004


The best embedding according to our metric is dim_512_neg_18_context_2 with a value of 0.1121, so we will stick with that for now. We now investigate:
learning_rate: [5e-4, 1e-3, 5e-3]
dropout: [0.1, 0.2, 0.3]
weight_decay: [1e-5, 1e-4, 1e-3]
label_smoothing: [0.0, 0.1]

In [10]:
from itertools import product
phase_dir = "gridSearch/optimization"
os.makedirs(phase_dir, exist_ok=True)

sweep_params = {
    'learning_rate': [0.001, 0.005, 0.01],
    'dropout': [0.1, 0.2, 0.3],
    'weight_decay': [1e-5, 1e-4, 1e-3],
    'label_smoothing': [0.0, 0.1, 0.2],
}
fixed_params = {
    'embedding_dim': 512,
    'num_negative': 18,
    'context_size': 2,
    'batch_size': 1024,
    'epochs': 20,
    'validation_fraction': 0.1,
    'exclude_all_contexts': True,
    'patience': 5,
    'device': None
}

for lr, dropout, wd, ls in product(
    sweep_params['learning_rate'],
    sweep_params['dropout'],
    sweep_params['weight_decay'],
    sweep_params['label_smoothing'],
):
    config = {
        **fixed_params,
        'learning_rate': lr,
        'dropout': dropout,
        'weight_decay': wd,
        'label_smoothing': ls,
    }
    print(f"Running: lr={lr}, dropout={dropout}, wd={wd}, ls={ls}")
    results = train_embeddings(
        network_data=network_data,
        **config,
        save_path=f"{phase_dir}/lr_{lr}_dropout_{dropout}_wd_{wd}_ls_{ls}.pth",
    )


Running: lr=0.001, dropout=0.1, wd=1e-05, ls=0.0

🔧 PUNCTUATION FILTER:
  Removed: {',', '<RARE>', '.', "'"}
  Nodes: 457 → 454
  Edges: 49,823 → 48,463

Train edges: 43,616, Val edges: 4,847

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 205,662
  Negatives per positive: 18
  Total samples per epoch: 3,907,578

  Weight distribution:
    Min: 0.028324
    Mean: 0.083439
    Median: 0.028324
    Max: 1.000000

📊 SkipGramDataset Statistics:
  Vocabulary size: 454
  Positive pairs: 138,490
  Negatives per positive: 18
  Total samples per epoch: 2,631,310

  Weight distribution:
    Min: 0.022813
    Mean: 0.079994
    Median: 0.027940
    Max: 1.000000


/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



Training on mps
Vocab: 454, Embed dim: 512, Context: 2, Negatives: 18
Regularization: dropout=0.1, weight_decay=1e-05, label_smoothing=0.0


Epoch 01  train=5.5657  val=3.8500  lr=0.001000
  → Best model (val_loss=3.8500), saved to gridSearch/optimization/lr_0.001_dropout_0.1_wd_1e-05_ls_0.0.pth


Epoch 02  train=3.7181  val=3.8473  lr=0.001000
  → Best model (val_loss=3.8473), saved to gridSearch/optimization/lr_0.001_dropout_0.1_wd_1e-05_ls_0.0.pth


...

Epoch 09  train=10.2498  val=6.0833  lr=0.005000


Epoch 10  train=10.2508  val=6.0659  lr=0.005000


Epoch 11  train=10.2507  val=6.0842  lr=0.005000


Epoch 12  train=10.2416  val=6.0737  lr=0.002500


Epoch 13  train=10.2375  val=6.0712  lr=0.002500

Early stopping at epoch 13

Saved loss plot to gridSearch/optimization/lr_0.01_dropout_0.3_wd_0.001_ls_0.2_training_loss.png


### Analysis

In [12]:
save_dir = "./gridSearch/optimization"
all_signal_to_noise = {}

for filename in sorted(os.listdir(save_dir)):
    if not filename.endswith(".pth"):
        continue
    # --- 1. Load Best Model ---
    # We load the model, nodes, and embeddings saved by train_embeddings
    model_path = os.path.join(save_dir, filename)
    prefix = os.path.splitext(filename)[0]  # Remove .pth extension
    output_txt = os.path.join(save_dir + "/results/", f"{prefix}.txt")

    print(f"\nProcessing {filename}")
    print(f"→ Saving output to {output_txt}")

    original_stdout = sys.stdout
    with open(output_txt, "w") as f:
        sys.stdout = f  # Change the standard output to the file we created.
        if os.path.exists(model_path):
            print(f"{save_dir}: Loading best model from {model_path}...")
            checkpoint = torch.load(model_path)
            nodes = checkpoint['nodes']

            # Reconstruct model to get final (non-detached) embeddings
            # Note: We use .get_embeddings() which returns the center_embeddings
            model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
            model.load_state_dict(checkpoint['model_state_dict'])
            embeddings = model.get_embeddings()

            print(f"✅ Loaded embeddings for {len(nodes):,} words")
        else:
            print("❌ Error: No saved model found!")
            print("Please run the training cell above before proceeding.")
            # Raise an error to stop execution if the model is missing
            raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

        # --- 2. Qualitative Analysis ---
        signal_to_noise_metric =  ranking_embeddings_signal_to_noise(nodes, embeddings)
        print(f"Signal to Noise Metric: {signal_to_noise_metric:.4f}")
        all_signal_to_noise[prefix] = signal_to_noise_metric
        # Now we use the helper functions to probe what the model learned.
        # We'll check for nearest neighbors (semantics) and analogies (linear structure).
        analyze_embeddings(
            nodes=nodes,
            embeddings=embeddings,

            # --- Nearest neighbors ---
            # Probe visual semantics, attributes, and compositional structure
            similarity_examples=[
                # People and roles
                "man", "woman", "child", "boy", "girl", "person",
                # Scene elements
                "tree", "building", "sky", "street", "car", "table",
                # Colors and materials
                "red", "blue", "green", "black", "white", "wooden", "metal",
                # Actions
                "walking", "sitting", "holding", "riding", "standing",
                # Textures / objects
                "water", "grass", "sand", "snow", "wall"
            ],

            # --- Analogies ---
            # a : b :: c : ?
            #  → test if embedding geometry captures color, role, size, material, and action relations
            analogy_examples=[
                # Color analogies
                ("red", "apple", "yellow"),        # → banana?
                ("blue", "sky", "green"),          # → grass?
                ("white", "snow", "brown"),        # → dirt or wood?

                # Role analogies
                ("man", "woman", "boy"),           # → girl?
                ("boy", "girl", "man"),            # → woman?
                ("person", "hat", "hand"),         # → glove?

                # Size / quantity / attribute analogies
                ("long", "short", "tall"),         # → short?
                ("one", "two", "three"),           # → four?

                # Action and object use
                ("dog", "walking", "cat"),         # → sitting?
                ("person", "riding", "boat"),      # → sitting or water?
                ("pizza", "eating", "umbrella"),   # → holding?

                # Material / context
                ("metal", "car", "wooden"),        # → table?
                ("water", "boat", "road"),         # → car?
            ],

            # --- Semantic clusters ---
            # seeds chosen to reveal structure in scene-object relationships
            cluster_seeds=[
                "red", "blue", "green",          # color cluster
                "man", "woman", "child",         # human cluster
                "dog", "cat", "horse", "bird",   # animal cluster
                "building", "house", "street",   # architecture/scene
                "sky", "clouds", "water", "grass", # natural elements
                "car", "bus", "bicycle", "train" # vehicles
            ]
        )

        # --- 3. t-SNE Visualization ---
        # We'll create a 2D plot of the *most frequent* words
        # NOTE: Visualizing all 10k+ words is too slow and unreadable.
        # The function defaults to the top 200, we'll use 300.
        print("\n" + "="*80)
        print("VISUALIZING EMBEDDINGS")
        print("="*80)
        print("Generating t-SNE plot for the top 300 words...")
        print("(This may take a moment...)")

        tsne_file = os.path.join(save_dir + "/results", f"{prefix}_tsne.png")
        visualize_embeddings(
            nodes,
            embeddings,
            output_file=tsne_file,
            sample_size=len(nodes),
            annotate=True
        )
    sys.stdout = original_stdout  # Reset standard output to original value


Processing lr_0.001_dropout_0.1_wd_0.0001_ls_0.0.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.001_dropout_0.1_wd_0.0001_ls_0.0.txt

Processing lr_0.001_dropout_0.1_wd_0.0001_ls_0.1.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.001_dropout_0.1_wd_0.0001_ls_0.1.txt

Processing lr_0.001_dropout_0.1_wd_0.0001_ls_0.2.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.001_dropout_0.1_wd_0.0001_ls_0.2.txt

Processing lr_0.001_dropout_0.1_wd_0.001_ls_0.0.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.001_dropout_0.1_wd_0.001_ls_0.0.txt

Processing lr_0.001_dropout_0.1_wd_0.001_ls_0.1.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.001_dropout_0.1_wd_0.001_ls_0.1.txt

Processing lr_0.001_dropout_0.1_wd_0.001_ls_0.2.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.001_dropout_0.1_wd_0.001_ls_0.2.txt

Processing lr_0.001_dropout_0.1_wd_1e-05_ls_0.0.pth
→ Saving output to ./gridSearch/optimization/results/lr_0.0

In [13]:
# Top 10 in all_mean_minus_p5, all_cifar_cls_sim -> see if there's anything in common
def top_k(d, k=10):
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:k]

top_signal_to_noise = top_k(all_signal_to_noise, k=20)

print("Top 10 Signal to Noise Metrics:")
for name, score in top_signal_to_noise:
    print(f"{name}: {score:.4f}")    

Top 10 Signal to Noise Metrics:
lr_0.001_dropout_0.3_wd_1e-05_ls_0.2: 0.2482
lr_0.001_dropout_0.3_wd_0.0001_ls_0.2: 0.2326
lr_0.001_dropout_0.3_wd_0.001_ls_0.2: 0.2215
lr_0.001_dropout_0.1_wd_0.0001_ls_0.2: 0.2102
lr_0.01_dropout_0.1_wd_1e-05_ls_0.0: 0.1763
lr_0.01_dropout_0.2_wd_0.0001_ls_0.0: 0.1723
lr_0.01_dropout_0.1_wd_0.001_ls_0.0: 0.1686
lr_0.005_dropout_0.1_wd_0.0001_ls_0.1: 0.1607
lr_0.01_dropout_0.1_wd_0.0001_ls_0.0: 0.1573
lr_0.005_dropout_0.2_wd_0.001_ls_0.1: 0.1569
lr_0.01_dropout_0.3_wd_0.001_ls_0.0: 0.1506
lr_0.005_dropout_0.3_wd_1e-05_ls_0.1: 0.1504
lr_0.01_dropout_0.3_wd_0.0001_ls_0.0: 0.1455
lr_0.005_dropout_0.1_wd_0.001_ls_0.1: 0.1416
lr_0.01_dropout_0.1_wd_1e-05_ls_0.1: 0.1402
lr_0.01_dropout_0.2_wd_0.001_ls_0.0: 0.1402
lr_0.005_dropout_0.2_wd_0.0001_ls_0.1: 0.1388
lr_0.005_dropout_0.1_wd_1e-05_ls_0.1: 0.1364
lr_0.01_dropout_0.1_wd_0.001_ls_0.1: 0.1332
lr_0.01_dropout_0.2_wd_1e-05_ls_0.0: 0.1301


# Training Best Existing Embedding Model

In [29]:
print("\n" + "="*80)
print("🚀 STARTING TRAINING RUN")
print("This may take a few minutes. We are running the full pipeline...")
print("="*80)

phase_dir = "."

best_config = {
    'embedding_dim': 512,
    'batch_size': 1024,
    'epochs': 20,
    'learning_rate': 0.001,
    'num_negative': 18,
    'validation_fraction':0.1,
    'context_size': 2,
    'dropout': 0.3,
    'weight_decay': 1e-5,
    'label_smoothing': 0.2,
    'exclude_all_contexts': True,
    'patience': 5,
    'device': None
}

results = train_embeddings(
    network_data=network_data,
    **best_config,
    save_path=f"{phase_dir}/best_embedding.pth",
)
# --- Training Summary ---
nodes = results['nodes']
embeddings = results['embeddings']

print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)
print(f"Learned embeddings for {len(nodes):,} words")
print(f"Embedding dimension: {embeddings.shape[1]}")


🚀 STARTING TRAINING RUN
This may take a few minutes. We are running the full pipeline...

🔧 PUNCTUATION FILTER:
  Removed: {',', '<RARE>', '.', "'"}
  Nodes: 458 → 455
  Edges: 50,128 → 48,764

Train edges: 43,887, Val edges: 4,877

📊 SkipGramDataset Statistics:
  Vocabulary size: 455
  Positive pairs: 206,570
  Negatives per positive: 18
  Total samples per epoch: 3,924,830

  Weight distribution:
    Min: 0.028427
    Mean: 0.083663
    Median: 0.028427
    Max: 1.000000

📊 SkipGramDataset Statistics:
  Vocabulary size: 455
  Positive pairs: 139,876
  Negatives per positive: 18
  Total samples per epoch: 2,657,644

  Weight distribution:
    Min: 0.022920
    Mean: 0.079899
    Median: 0.028072
    Max: 1.000000


/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



Training on mps
Vocab: 455, Embed dim: 512, Context: 2, Negatives: 18
Regularization: dropout=0.3, weight_decay=1e-05, label_smoothing=0.2


Epoch 01  train=10.6859  val=6.0694  lr=0.001000
  → Best model (val_loss=6.0694), saved to ./best_embedding.pth


Epoch 02  train=10.2546  val=6.0647  lr=0.001000
  → Best model (val_loss=6.0647), saved to ./best_embedding.pth


...

Epoch 12  train=10.2275  val=6.0543  lr=0.000500


Epoch 13  train=10.2263  val=6.0476  lr=0.000500


Epoch 14  train=10.2267  val=6.0524  lr=0.000500


Epoch 15  train=10.2251  val=6.0481  lr=0.000250


Epoch 16  train=10.2256  val=6.0517  lr=0.000250

Early stopping at epoch 16

Saved loss plot to ./best_embedding_training_loss.png

✅ TRAINING COMPLETE
Learned embeddings for 455 words
Embedding dimension: 512


In [3]:
save_dir = "."
all_signal_to_noise = {}

filename = "best_embedding.pth"
# --- 1. Load Best Model ---
# We load the model, nodes, and embeddings saved by train_embeddings
model_path = os.path.join(save_dir, filename)
prefix = os.path.splitext(filename)[0]  # Remove .pth extension
output_txt = os.path.join(save_dir, f"{prefix}.txt")

print(f"\nProcessing {filename}")
print(f"→ Saving output to {output_txt}")

original_stdout = sys.stdout
with open(output_txt, "w") as f:
    sys.stdout = f  # Change the standard output to the file we created.
    if os.path.exists(model_path):
        print(f"{save_dir}: Loading best model from {model_path}...")
        checkpoint = torch.load(model_path)
        nodes = checkpoint['nodes']

        # Reconstruct model to get final (non-detached) embeddings
        # Note: We use .get_embeddings() which returns the center_embeddings
        model = SkipGramModel(checkpoint['vocab_size'], checkpoint['embedding_dim'])
        model.load_state_dict(checkpoint['model_state_dict'])
        embeddings = model.get_embeddings()

        print(f"✅ Loaded embeddings for {len(nodes):,} words")
    else:
        print("❌ Error: No saved model found!")
        print("Please run the training cell above before proceeding.")
        # Raise an error to stop execution if the model is missing
        raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

    # --- 2. Qualitative Analysis ---
    signal_to_noise_metric =  ranking_embeddings_signal_to_noise(nodes, embeddings)
    print(f"Signal to Noise Metric: {signal_to_noise_metric:.4f}")
    all_signal_to_noise[prefix] = signal_to_noise_metric
    # Now we use the helper functions to probe what the model learned.
    # We'll check for nearest neighbors (semantics) and analogies (linear structure).
    analyze_embeddings(
        nodes=nodes,
        embeddings=embeddings,

        # --- Nearest neighbors ---
        # Probe visual semantics, attributes, and compositional structure
        similarity_examples=[
            # People and roles
            "man", "woman", "child", "boy", "girl", "person",
            # Scene elements
            "tree", "building", "sky", "street", "car", "table",
            # Colors and materials
            "red", "blue", "green", "black", "white", "wooden", "metal",
            # Actions
            "walking", "sitting", "holding", "riding", "standing",
            # Textures / objects
            "water", "grass", "sand", "snow", "wall"
        ],

        # --- Analogies ---
        # a : b :: c : ?
        #  → test if embedding geometry captures color, role, size, material, and action relations
        analogy_examples=[
            # Color analogies
            ("red", "apple", "yellow"),        # → banana?
            ("blue", "sky", "green"),          # → grass?
            ("white", "snow", "brown"),        # → dirt or wood?

            # Role analogies
            ("man", "woman", "boy"),           # → girl?
            ("boy", "girl", "man"),            # → woman?
            ("person", "hat", "hand"),         # → glove?

            # Size / quantity / attribute analogies
            ("long", "short", "tall"),         # → short?
            ("one", "two", "three"),           # → four?

            # Action and object use
            ("dog", "walking", "cat"),         # → sitting?
            ("person", "riding", "boat"),      # → sitting or water?
            ("pizza", "eating", "umbrella"),   # → holding?

            # Material / context
            ("metal", "car", "wooden"),        # → table?
            ("water", "boat", "road"),         # → car?
        ],

        # --- Semantic clusters ---
        # seeds chosen to reveal structure in scene-object relationships
        cluster_seeds=[
            "red", "blue", "green",          # color cluster
            "man", "woman", "child",         # human cluster
            "dog", "cat", "horse", "bird",   # animal cluster
            "building", "house", "street",   # architecture/scene
            "sky", "clouds", "water", "grass", # natural elements
            "car", "bus", "bicycle", "train" # vehicles
        ]
    )

    # --- 3. t-SNE Visualization ---
    # We'll create a 2D plot of the *most frequent* words
    # NOTE: Visualizing all 10k+ words is too slow and unreadable.
    # The function defaults to the top 200, we'll use 300.
    print("\n" + "="*80)
    print("VISUALIZING EMBEDDINGS")
    print("="*80)
    print("Generating t-SNE plot for the top 300 words...")
    print("(This may take a moment...)")

    tsne_file = os.path.join(save_dir, f"{prefix}_tsne.png")
    visualize_embeddings(
        nodes,
        embeddings,
        output_file=tsne_file,
        sample_size=len(nodes),
        annotate=True
    )
sys.stdout = original_stdout  # Reset standard output to original valuesa


Processing best_embedding.pth
→ Saving output to ./best_embedding.txt


In [4]:
print(nodes)

['a', 'the', 'on', 'of', 'is', 'in', 'white', 'black', 'and', 'man', 'with', 'blue', 'red', 'green', 'wearing', 'brown', 'building', 'are', 'person', 'woman', 'this', 'wall', 'sky', 'window', 'yellow', 'shirt', 'sign', 'water', 'table', 'to', 'has', 'tree', 'light', 'train', 'two', 'grass', 'an', 'side', 'large', 'small', 'street', 'front', 'ground', 'top', 'plate', 'car', 'part', 'orange', 'head', 'clouds', 'wooden', 'standing', 'bus', 'pole', 'sitting', 'metal', 'behind', 'holding', 'color', 'trees', 'silver', 'snow', 'gray', 'people', 'dog', 'hand', 'road', 'tennis', 'hair', 'grey', 'dark', 'glass', 'at', 'plane', 'back', 'floor', 'cat', 'background', 'fence', 'clock', 'door', 'giraffe', 'leaves', 'boy', 'left', 'field', 'right', 'next', 'long', 'by', 'bear', 'chair', 'elephant', 'tall', 'pink', "man's", 'girl', 'horse', 'pizza', 'for', 'baseball', 'zebra', 'pants', 'hat', 'boat', 'truck', 'walking', 'jacket', 'bench', 'sidewalk', 'it', 'hanging', 'toilet', 'up', 'bird', 'tail', 'be